In [ ]:
import csv
import io
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

MIPT_PATH = 'MIPT_hackathon_dataset_sla_hackathon.csv'
EXTERNAL_PATH = 'dataset_2025-03-01_2026-03-29_external (Новый датасет 30.03).csv'
OUTPUT_DIR = Path('sla_prep_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = [
    'lead_id',
    'sale_date',
    'lead_created_at',
    'sale_ts',
    'lead_Дата перехода в Сборку',
    'handed_to_delivery_ts',
    'lead_Дата перехода Передан в доставку',
    'issued_or_pvz_ts',
    'received_ts',
    'rejected_ts',
    'returned_ts',
    'closed_ts',
    'lead_pipeline_id',
    'current_status_id',
    'lead_responsible_user_id',
    'lead_Ответственный за доставку',
    'lifecycle_incomplete',
    'buyout_flag',
    'outcome_unknown',
    'lead_group_id',
    'lead_group',
    'contact_Город',
    'lead_Служба доставки',
    'lead_Метод доставки',
    'lead_Квалификация лида',
    'lead_Вид оплаты',
    'lead_source',
    'lead_Источник',
    'lead_Тариф Доставки',
    'lead_loss_reason_id',
]

RENAME_MAP = {
    'contact_Город': 'region',
    'lead_Служба доставки': 'delivery_service',
    'lead_Тариф Доставки': 'delivery_tariff',
    'lead_Вид оплаты': 'payment_type',
    'lead_loss_reason_id': 'rejection_category',
    'lead_responsible_user_id': 'manager_id',
    'lead_Ответственный за доставку': 'delivery_manager_id',
    'lead_Метод доставки': 'delivery_method',
    'lead_Квалификация лида': 'lead_qualification',
    'lead_group_id': 'manager_group_id',
    'lead_group': 'manager_group',
}

TIME_COLUMNS = [
    'lead_created_at',
    'sale_ts',
    'lead_Дата перехода в Сборку',
    'handed_to_delivery_ts',
    'lead_Дата перехода Передан в доставку',
    'issued_or_pvz_ts',
    'received_ts',
    'rejected_ts',
    'returned_ts',
    'closed_ts',
]

BOOL_COLUMNS = ['lifecycle_incomplete', 'buyout_flag', 'outcome_unknown']


In [ ]:
def normalize_text_columns(columns):
    cleaned = []
    for c in columns:
        c = str(c).strip().strip('"')
        c = c.replace('\ufeff', '')
        cleaned.append(c)
    return cleaned


def parse_bool_series(series: pd.Series) -> pd.Series:
    mapping = {
        'true': True,
        'false': False,
        '1': True,
        '0': False,
        'yes': True,
        'no': False,
        '': pd.NA,
        'nan': pd.NA,
        'none': pd.NA,
    }
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
        .astype('boolean')
    )


def parse_unix_ts(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors='coerce')
    return pd.to_datetime(numeric, unit='s', errors='coerce')


def ensure_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def coalesce_columns(base: pd.Series, other: pd.Series) -> pd.Series:
    return base.where(base.notna(), other)


In [ ]:
def parse_external_multiline_csv(path: str) -> pd.DataFrame:
    text = Path(path).read_text(encoding='utf-8-sig', errors='replace')
    parts = text.split(';;;')

    header_chunk = parts[0].strip()
    record_starts = []
    for i, part in enumerate(parts):
        s = part.strip()
        if s.startswith('"LEAD_') or s.startswith('LEAD_'):
            record_starts.append(i)

    if not record_starts:
        raise ValueError('Не удалось найти записи LEAD_ во внешнем файле')

    def make_csv_line(raw: str) -> str:
        raw = raw.replace('\r', ' ').replace('\n', ' ')
        raw = raw.replace('""', '"')
        raw = raw.rstrip('"').strip()
        if raw.startswith('"'):
            raw = raw[1:]
        return raw

    header_line = make_csv_line(header_chunk)
    header = next(csv.reader(io.StringIO(header_line)))
    header = normalize_text_columns(header)

    records = []
    for idx, start in enumerate(record_starts):
        end = record_starts[idx + 1] if idx + 1 < len(record_starts) else len(parts)
        raw_record = ';;;'.join(parts[start:end])
        line = make_csv_line(raw_record)
        row = next(csv.reader(io.StringIO(line)))
        if len(row) < len(header):
            row = row + [''] * (len(header) - len(row))
        elif len(row) > len(header):
            row = row[:len(header)]
        records.append(row)

    df = pd.DataFrame(records, columns=header)
    df.columns = normalize_text_columns(df.columns)
    return df


In [ ]:
def standardize_dataframe(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = normalize_text_columns(df.columns)
    df = ensure_columns(df, TARGET_COLUMNS)

    for col in BOOL_COLUMNS:
        if col in df.columns:
            df[col] = parse_bool_series(df[col])

    for col in TIME_COLUMNS:
        if col in df.columns:
            df[col] = parse_unix_ts(df[col])

    if 'sale_date' in df.columns:
        df['sale_date'] = pd.to_datetime(df['sale_date'], errors='coerce').dt.date

    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})

    df['source_dataset'] = source_name
    return df


In [ ]:
df_mipt_raw = pd.read_csv(MIPT_PATH)
df_ext_raw = parse_external_multiline_csv(EXTERNAL_PATH)

print('MIPT raw shape:', df_mipt_raw.shape)
print('External raw shape:', df_ext_raw.shape)

display(df_mipt_raw.head(2))
display(df_ext_raw.head(2))


In [ ]:
df_mipt = standardize_dataframe(df_mipt_raw, 'mipt')
df_ext = standardize_dataframe(df_ext_raw, 'external')

print('MIPT standardized:', df_mipt.shape)
print('External standardized:', df_ext.shape)

display(df_mipt[TARGET_COLUMNS[:10]].head(2))
display(df_ext[TARGET_COLUMNS[:10]].head(2))


In [ ]:
def merge_mipt_with_external(df_mipt: pd.DataFrame, df_ext: pd.DataFrame) -> pd.DataFrame:
    ext = df_ext.copy().drop_duplicates(subset=['lead_id'], keep='first')
    mipt = df_mipt.copy().drop_duplicates(subset=['lead_id'], keep='first')

    merged = ext.merge(
        mipt[TARGET_COLUMNS],
        on='lead_id',
        how='left',
        suffixes=('', '__mipt')
    )

    for col in TARGET_COLUMNS:
        if col == 'lead_id':
            continue
        mipt_col = f'{col}__mipt'
        if mipt_col in merged.columns:
            merged[col] = coalesce_columns(merged[col], merged[mipt_col])

    drop_cols = [c for c in merged.columns if c.endswith('__mipt')]
    merged = merged.drop(columns=drop_cols)
    return merged


df_merged = merge_mipt_with_external(df_mipt, df_ext)
print('Merged shape:', df_merged.shape)
display(df_merged.head(3))


In [ ]:
def clean_prepared_data(df: pd.DataFrame):
    df = df.copy()
    stats = {}
    stats['rows_before_cleaning'] = len(df)

    mask_lifecycle = df['lifecycle_incomplete'].fillna(False)
    stats['excluded_lifecycle_incomplete'] = int(mask_lifecycle.sum())
    df = df.loc[~mask_lifecycle].copy()

    h1 = df['handed_to_delivery_ts']
    h2 = df['lead_Дата перехода Передан в доставку']
    mismatch_mask = h1.notna() & h2.notna() & (h1 != h2)
    stats['handover_ts_mismatch_count'] = int(mismatch_mask.sum())

    df['speed_to_delivery_days'] = (
        (df['handed_to_delivery_ts'] - df['sale_ts']).dt.total_seconds() / 86400
    )

    anomaly_masks = {
        'sale_before_creation': df['sale_ts'].notna() & df['lead_created_at'].notna() & (df['sale_ts'] < df['lead_created_at']),
        'assembly_before_sale': df['lead_Дата перехода в Сборку'].notna() & df['sale_ts'].notna() & (df['lead_Дата перехода в Сборку'] < df['sale_ts']),
        'delivery_before_assembly': df['handed_to_delivery_ts'].notna() & df['lead_Дата перехода в Сборку'].notna() & (df['handed_to_delivery_ts'] < df['lead_Дата перехода в Сборку']),
        'speed_to_delivery_negative': df['speed_to_delivery_days'].notna() & (df['speed_to_delivery_days'] < 0),
    }

    union_anomaly = pd.Series(False, index=df.index)
    for name, mask in anomaly_masks.items():
        stats[name] = int(mask.sum())
        union_anomaly = union_anomaly | mask

    stats['excluded_anomalies_total'] = int(union_anomaly.sum())
    df = df.loc[~union_anomaly].copy()

    categorical_fill = {
        'contact_Город': 'Не указан',
        'lead_Служба доставки': 'Не указана',
        'lead_Тариф Доставки': 'Не указан',
        'lead_Вид оплаты': 'Не указан',
        'lead_loss_reason_id': 'Нет отказа / не указан',
        'lead_responsible_user_id': 'Не указан',
        'lead_Ответственный за доставку': 'Не указан',
        'lead_Метод доставки': 'Не указан',
        'lead_Квалификация лида': 'Не указана',
        'lead_group': 'Не указана',
        'lead_group_id': 'Не указан',
        'lead_source': 'Не указан',
        'lead_Источник': 'Не указан',
    }
    for col, filler in categorical_fill.items():
        if col in df.columns:
            df[col] = df[col].fillna(filler)

    stats['rows_after_cleaning'] = len(df)
    return df, stats


df_clean, cleaning_stats = clean_prepared_data(df_merged)
print(json.dumps(cleaning_stats, ensure_ascii=False, indent=2, default=str))
display(df_clean.head(3))


In [ ]:
df_analytics = df_clean.rename(columns=RENAME_MAP).copy()

analytics_columns_order = [
    'lead_id',
    'sale_date',
    'lead_created_at',
    'sale_ts',
    'lead_Дата перехода в Сборку',
    'handed_to_delivery_ts',
    'lead_Дата перехода Передан в доставку',
    'issued_or_pvz_ts',
    'received_ts',
    'rejected_ts',
    'returned_ts',
    'closed_ts',
    'lead_pipeline_id',
    'current_status_id',
    'lifecycle_incomplete',
    'buyout_flag',
    'outcome_unknown',
    'region',
    'delivery_service',
    'delivery_tariff',
    'payment_type',
    'delivery_method',
    'lead_qualification',
    'rejection_category',
    'manager_id',
    'delivery_manager_id',
    'manager_group_id',
    'manager_group',
    'speed_to_delivery_days',
    'lead_source',
    'lead_Источник',
    'source_dataset',
]

analytics_columns_order = [c for c in analytics_columns_order if c in df_analytics.columns]
df_analytics = df_analytics[analytics_columns_order].copy()

print(df_analytics.shape)
display(df_analytics.head(5))


In [ ]:
missing_profile = pd.DataFrame(
    {
        'column': df_clean.columns,
        'null_count': [df_clean[c].isna().sum() for c in df_clean.columns],
        'null_percent': [round(df_clean[c].isna().mean() * 100, 2) for c in df_clean.columns],
        'dtype': [str(df_clean[c].dtype) for c in df_clean.columns],
    }
).sort_values(['null_percent', 'column'], ascending=[False, True])

top_regions = (
    df_clean['contact_Город']
    .value_counts(dropna=False)
    .rename_axis('region')
    .reset_index(name='orders')
    .head(10)
)

top_delivery_services = (
    df_clean['lead_Служба доставки']
    .value_counts(dropna=False)
    .rename_axis('delivery_service')
    .reset_index(name='orders')
    .head(10)
)

display(missing_profile.head(20))
display(top_regions)
display(top_delivery_services)


In [ ]:
plt.figure(figsize=(12, 7))
plt.bar(top_delivery_services['delivery_service'].astype(str), top_delivery_services['orders'])
plt.xticks(rotation=45, ha='right')
plt.xlabel('Служба доставки')
plt.ylabel('Количество заказов')
plt.title('Топ-10 служб доставки по числу заказов')
plt.tight_layout()
plt.show()


In [ ]:
prepared_path = OUTPUT_DIR / 'prepared_stage1_dataset.csv'
profile_path = OUTPUT_DIR / 'profile_missing_values.csv'
regions_path = OUTPUT_DIR / 'top_regions.csv'
services_path = OUTPUT_DIR / 'top_delivery_services.csv'
stats_path = OUTPUT_DIR / 'cleaning_stats.json'
chart_path = OUTPUT_DIR / 'delivery_services_top10.png'

df_analytics.to_csv(prepared_path, index=False)
missing_profile.to_csv(profile_path, index=False)
top_regions.to_csv(regions_path, index=False)
top_delivery_services.to_csv(services_path, index=False)
stats_path.write_text(json.dumps(cleaning_stats, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

plt.figure(figsize=(12, 7))
plt.bar(top_delivery_services['delivery_service'].astype(str), top_delivery_services['orders'])
plt.xticks(rotation=45, ha='right')
plt.xlabel('Служба доставки')
plt.ylabel('Количество заказов')
plt.title('Топ-10 служб доставки по числу заказов')
plt.tight_layout()
plt.savefig(chart_path, dpi=200, bbox_inches='tight')
plt.close()

print('Файлы сохранены:')
print(prepared_path)
print(profile_path)
print(regions_path)
print(services_path)
print(stats_path)
print(chart_path)
